# 5-Qubit Linear Chain Chip - Eigenmode Simulation


In [ ]:
!git clone https://github.com/Abdelwhabmohammed/cinqubit.git
%cd cinqubit/src
!pip install qdesignoptimizer
!pip install "quantum-metal[full]"

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Rendering the Design

In [ ]:
import names as n
import design as d
from qdesignoptimizer.utils.chip_generation import create_chip_base

design, gui = create_chip_base(n.CHIP_NAME, d.chip_type, open_gui=False)
d.render_qiskit_metal_design(design, gui)

## 2. Design Rule Check (DRC)


In [ ]:
import design_rules as dr

violations = dr.run_drc(
    qubit_positions=d.QUBIT_POSITIONS,
    chip_size_x_mm=14.0,
    chip_size_y_mm=14.0,
)
dr.print_drc_report(violations)

## 3. Parameter Sweep — Qubit Pitch Optimization


In [ ]:
import parameter_sweep as ps
import numpy as np

sweep_data = ps.sweep_qubit_pitch(
    pitch_range_um=(1600, 3500),
    n_steps=20,
    penalty_weight=2.0,
    chip_size_x_mm=14.0,
    chip_size_y_mm=14.0,
)

optimal_idx = np.argmin(sweep_data['costs'])
print(f"Optimal pitch: {sweep_data['pitches_um'][optimal_idx]:.0f} um")
print(f"Minimum cost: {sweep_data['costs'][optimal_idx]:.0f}")

ps.plot_sweep_results(sweep_data, save_path="out/pitch_sweep_linear.png")

## 4. Creating Study and Optimization Targets

In [ ]:
import mini_studies as ms
import optimization_targets as ot

MINI_STUDY_GROUP = n.ALL_GROUPS
MINI_STUDY = ms.get_mini_study_5qb_resonator_coupler()
RENDER_QISKIT_METAL = lambda design: d.render_qiskit_metal_design(design, gui)

opt_targets = ot.get_opt_targets_5qubits_resonator_coupler(
    groups=MINI_STUDY_GROUP,
    opt_target_qubit_freq=True,
    opt_target_qubit_anharm=True,
    opt_target_resonator_freq=True,
    opt_target_resonator_kappa=True,
    opt_target_resonator_qubit_chi=True,
    opt_target_coupler_freq=True,
    use_simple_resonator_qubit_chi_relation=True,
)

print(f"Total optimization targets: {len(opt_targets)}")

## 5. Creating Design Analysis Objects


In [ ]:
import parameter_targets as pt
import plot_settings as pls

from qdesignoptimizer.design_analysis import DesignAnalysis, DesignAnalysisState
from qdesignoptimizer.utils.utils import get_save_path, close_ansys

# Close any existing Ansys session
close_ansys()

design_analysis_state = DesignAnalysisState(
    design, RENDER_QISKIT_METAL, pt.PARAM_TARGETS
)

design_analysis = DesignAnalysis(
    design_analysis_state,
    mini_study=MINI_STUDY,
    opt_targets=opt_targets,
    save_path=get_save_path("out/", n.CHIP_NAME),
    update_design_variables=False,
    plot_settings=pls.PLOT_SETTINGS_5QB,
)

## 6. Optimization Loop

In [ ]:
nbr_iterations = 10
group_passes_start = 8
group_passes_stop = 15
delta_f = 0.001

for i in range(nbr_iterations):
    design_analysis.update_nbr_passes(
        min(group_passes_start + i, group_passes_stop)
    )
    design_analysis.update_delta_f(delta_f)
    design_analysis.optimize_target({}, {})
    design_analysis.screenshot(gui=gui, run=i)

## 7. Results

In [ ]:
# Eigenmodes
design_analysis.get_eigenmode_results()

In [ ]:
# Cross-Kerr matrix
design_analysis.get_cross_kerr_matrix(iteration=-1)

## 8. Topology Comparison

In [ ]:
import comparison as comp

candidates = [comp.LINEAR_CHAIN, comp.STAR_HUB]
df = comp.build_comparison_table(candidates)
if df is not None:
    display(df)

print()
print(comp.make_recommendation(candidates))

## 9. Update Parameters & Close

In [ ]:
# Overwrite design_variables.json with optimized values
design_analysis.overwrite_parameters()

In [ ]:
close_ansys()